# 第六章：GraphRAG 图增强检索

## 学习目标

- 理解 GraphRAG 相比纯向量 RAG 的优势
- 实现基于图结构的检索策略
- 构建混合检索（向量 + 图遍历）
- 实现实体提取驱动的图查询
- 对比纯向量 RAG vs GraphRAG 的回答质量

> **前置知识**：本章假设你已完成 LangChain 第五章（RAG）和 Neo4j 前五章的学习。我们将把 LangChain 中学到的向量检索与 Neo4j 的图查询结合起来。

## 0. 环境准备

加载环境变量、初始化 LLM 和 Neo4j 连接，然后构建一个科技公司知识图谱。

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

# 切换为 GLM（Anthropic 兼容接口）：
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="glm-4-plus", base_url=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"])

In [ ]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

print("Neo4j 连接成功")

### 构建科技公司知识图谱

为了保证数据一致性和可复现，我们用手动创建的方式写入预定义数据（第五章我们学过 `LLMGraphTransformer`，这里选择手动方式更可靠）。

图谱包含以下实体和关系：

| 节点类型 | 实例 |
|---------|------|
| Company | 阿里巴巴、腾讯、字节跳动、华为 |
| Person  | 马云、马化腾、张一鸣、任正非 |
| Product | 淘宝、支付宝、微信、QQ、抖音、TikTok、鸿蒙、华为云 |
| City    | 杭州、深圳、北京 |

| 关系类型 | 含义 |
|---------|------|
| FOUNDED | 人物 → 公司（创立） |
| CEO_OF  | 人物 → 公司（CEO） |
| HAS_PRODUCT | 公司 → 产品 |
| LOCATED_IN | 公司 → 城市 |
| COMPETES_WITH | 公司 ↔ 公司（竞争关系） |

In [ ]:
# 清空数据库（仅限学习环境！）
graph.query("MATCH (n) DETACH DELETE n")
print("已清空数据库")

In [ ]:
# 创建节点和关系
create_query = """
// 城市
CREATE (hangzhou:City {name: '杭州'})
CREATE (shenzhen:City {name: '深圳'})
CREATE (beijing:City {name: '北京'})

// 公司
CREATE (alibaba:Company {name: '阿里巴巴', founded_year: 1999, industry: '电商/云计算'})
CREATE (tencent:Company {name: '腾讯', founded_year: 1998, industry: '社交/游戏'})
CREATE (bytedance:Company {name: '字节跳动', founded_year: 2012, industry: '短视频/信息分发'})
CREATE (huawei:Company {name: '华为', founded_year: 1987, industry: '通信/消费电子'})

// 人物
CREATE (mayun:Person {name: '马云', title: '创始人'})
CREATE (mahuateng:Person {name: '马化腾', title: '创始人兼CEO'})
CREATE (zhangyiming:Person {name: '张一鸣', title: '创始人'})
CREATE (renzhengfei:Person {name: '任正非', title: '创始人兼CEO'})

// 产品
CREATE (taobao:Product {name: '淘宝', type: '电商平台', launch_year: 2003})
CREATE (alipay:Product {name: '支付宝', type: '支付平台', launch_year: 2004})
CREATE (wechat:Product {name: '微信', type: '社交平台', launch_year: 2011})
CREATE (qq:Product {name: 'QQ', type: '社交平台', launch_year: 1999})
CREATE (douyin:Product {name: '抖音', type: '短视频平台', launch_year: 2016})
CREATE (tiktok:Product {name: 'TikTok', type: '短视频平台', launch_year: 2017})
CREATE (hongmeng:Product {name: '鸿蒙', type: '操作系统', launch_year: 2019})
CREATE (huaweicloud:Product {name: '华为云', type: '云计算', launch_year: 2017})

// 关系：FOUNDED
CREATE (mayun)-[:FOUNDED {year: 1999}]->(alibaba)
CREATE (mahuateng)-[:FOUNDED {year: 1998}]->(tencent)
CREATE (zhangyiming)-[:FOUNDED {year: 2012}]->(bytedance)
CREATE (renzhengfei)-[:FOUNDED {year: 1987}]->(huawei)

// 关系：CEO_OF
CREATE (mahuateng)-[:CEO_OF]->(tencent)
CREATE (renzhengfei)-[:CEO_OF]->(huawei)

// 关系：HAS_PRODUCT
CREATE (alibaba)-[:HAS_PRODUCT]->(taobao)
CREATE (alibaba)-[:HAS_PRODUCT]->(alipay)
CREATE (tencent)-[:HAS_PRODUCT]->(wechat)
CREATE (tencent)-[:HAS_PRODUCT]->(qq)
CREATE (bytedance)-[:HAS_PRODUCT]->(douyin)
CREATE (bytedance)-[:HAS_PRODUCT]->(tiktok)
CREATE (huawei)-[:HAS_PRODUCT]->(hongmeng)
CREATE (huawei)-[:HAS_PRODUCT]->(huaweicloud)

// 关系：LOCATED_IN
CREATE (alibaba)-[:LOCATED_IN]->(hangzhou)
CREATE (tencent)-[:LOCATED_IN]->(shenzhen)
CREATE (bytedance)-[:LOCATED_IN]->(beijing)
CREATE (huawei)-[:LOCATED_IN]->(shenzhen)

// 关系：COMPETES_WITH（双向）
CREATE (alibaba)-[:COMPETES_WITH]->(tencent)
CREATE (tencent)-[:COMPETES_WITH]->(alibaba)
CREATE (bytedance)-[:COMPETES_WITH]->(tencent)
CREATE (tencent)-[:COMPETES_WITH]->(bytedance)
CREATE (alibaba)-[:COMPETES_WITH]->(bytedance)
CREATE (bytedance)-[:COMPETES_WITH]->(alibaba)
"""

graph.query(create_query)
print("知识图谱创建完成！")

In [ ]:
# 验证图谱
stats = graph.query("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY count DESC
""")
print("节点统计：")
for s in stats:
    print(f"  {s['label']}: {s['count']}")

rel_stats = graph.query("""
    MATCH ()-[r]->()
    RETURN type(r) AS type, count(r) AS count
    ORDER BY count DESC
""")
print("\n关系统计：")
for s in rel_stats:
    print(f"  {s['type']}: {s['count']}")

### 准备文本文档（供向量 RAG 使用）

同一批知识，也以非结构化文本的形式准备好，后面用来对比纯向量 RAG 和 GraphRAG 的效果。

In [ ]:
from langchain_core.documents import Document as LCDocument

# 将图谱知识转写为自然语言文档
docs = [
    LCDocument(
        page_content="阿里巴巴由马云于1999年在杭州创立，是一家电商和云计算公司。"
        "旗下主要产品包括淘宝（电商平台，2003年上线）和支付宝（支付平台，2004年上线）。"
        "阿里巴巴与腾讯、字节跳动存在竞争关系。",
        metadata={"company": "阿里巴巴"},
    ),
    LCDocument(
        page_content="腾讯由马化腾于1998年在深圳创立，马化腾同时担任CEO。"
        "腾讯是一家社交和游戏公司，旗下主要产品包括微信（社交平台，2011年上线）和QQ（社交平台，1999年上线）。"
        "腾讯与阿里巴巴、字节跳动存在竞争关系。",
        metadata={"company": "腾讯"},
    ),
    LCDocument(
        page_content="字节跳动由张一鸣于2012年在北京创立，是一家短视频和信息分发公司。"
        "旗下主要产品包括抖音（短视频平台，2016年上线）和TikTok（短视频平台，2017年上线）。"
        "字节跳动与腾讯、阿里巴巴存在竞争关系。",
        metadata={"company": "字节跳动"},
    ),
    LCDocument(
        page_content="华为由任正非于1987年在深圳创立，任正非同时担任CEO。"
        "华为是一家通信和消费电子公司，旗下主要产品包括鸿蒙（操作系统，2019年发布）和华为云（云计算，2017年上线）。",
        metadata={"company": "华为"},
    ),
    LCDocument(
        page_content="杭州是阿里巴巴的总部所在地，也是中国电商产业的重要城市。"
        "深圳是腾讯和华为的总部所在地，被称为中国科技之都。"
        "北京是字节跳动的总部所在地，也是中国互联网创业的核心城市。",
        metadata={"topic": "城市与公司"},
    ),
]

print(f"准备了 {len(docs)} 个文档，用于向量 RAG")

### 刚才发生了什么？

我们搭建了两套平行的知识存储：

1. **知识图谱**（Neo4j）：结构化的实体和关系，精确表达"谁创立了什么"、"总部在哪"等信息
2. **文本文档**（即将存入向量库）：同样的知识，但以自然语言段落的形式存在

后面我们会看到，同样的问题用不同的检索方式，效果截然不同。

---

## 1. 纯向量 RAG 的局限

在 LangChain 第五章中，我们学过用向量数据库做 RAG：把文档切片 → 嵌入向量 → 语义相似度检索 → 作为上下文交给 LLM。这个方案对很多场景够用了，但碰到**关系推理**类问题时，它的短板就暴露了。

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 初始化 Embedding 模型
embeddings = OpenAIEmbeddings(
    model="text-embedding-v3",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

# 构建向量索引
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("向量索引构建完成")

In [ ]:
# 测试语义检索
results = retriever.invoke("腾讯是谁创立的？")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:80]}...")

语义检索可以找到包含"腾讯"和"创立"的段落，对于这种简单的事实问题还行。但如果问题涉及**跨实体关系推理**呢？

In [ ]:
# ❌ 纯向量 RAG 回答关系推理问题
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "基于以下内容回答问题。如果内容不足以准确回答，请说明。\n\n"
    "内容：{context}\n\n"
    "问题：{question}"
)

chain = prompt | llm | StrOutputParser()

# 关系推理问题
question = "和腾讯在同一个城市的科技公司有哪些？"
context_docs = retriever.invoke(question)
context_text = "\n".join([d.page_content for d in context_docs])

answer = chain.invoke({"context": context_text, "question": question})
print(f"Q: {question}")
print(f"纯向量 RAG 回答: {answer}")
print(f"\n检索到的文档片段:")
for i, doc in enumerate(context_docs):
    print(f"  [{i+1}] {doc.page_content[:60]}...")

In [ ]:
# ❌ 再试一个多跳推理问题
question2 = "马云创立的公司和马化腾创立的公司之间是什么关系？"
context_docs2 = retriever.invoke(question2)
context_text2 = "\n".join([d.page_content for d in context_docs2])

answer2 = chain.invoke({"context": context_text2, "question": question2})
print(f"Q: {question2}")
print(f"纯向量 RAG 回答: {answer2}")

### 刚才发生了什么？

纯向量 RAG 的问题在于：

1. **语义检索只能找到"相关文本"**——问"和腾讯同城的公司"，检索器会找到含"腾讯"或"城市"的段落，但不一定能精确匹配到"同一个城市"的逻辑
2. **关系推理依赖 LLM 自行推断**——即使检索到了正确的段落，LLM 还需要从非结构化文本中抽取"腾讯在深圳"和"华为在深圳"这两个事实，然后做推理
3. **k 值有限**——检索 top-2 可能漏掉关键信息，增大 k 又引入噪声

这类"先定位实体、再沿关系查找"的问题，正是图数据库的强项。

---

## 2. 基于图的结构化检索

图数据库天然擅长关系查询。同样的问题，一句 Cypher 就能精确回答。

In [ ]:
# ✅ 图查询直接回答关系型问题
question = "和腾讯在同一个城市的科技公司有哪些？"

result = graph.query("""
    MATCH (c1:Company {name: '腾讯'})-[:LOCATED_IN]->(city:City)<-[:LOCATED_IN]-(c2:Company)
    WHERE c1 <> c2
    RETURN c2.name AS company, city.name AS city
""")

print(f"Q: {question}")
print("图查询结果：")
for r in result:
    print(f"  {r['company']}（{r['city']}）")

In [ ]:
# ✅ 多跳关系查询也很轻松
question2 = "马云创立的公司和马化腾创立的公司之间是什么关系？"

result2 = graph.query("""
    MATCH (p1:Person {name: '马云'})-[:FOUNDED]->(c1:Company)
    MATCH (p2:Person {name: '马化腾'})-[:FOUNDED]->(c2:Company)
    MATCH (c1)-[r:COMPETES_WITH]->(c2)
    RETURN p1.name AS person1, c1.name AS company1,
           type(r) AS relation,
           p2.name AS person2, c2.name AS company2
""")

print(f"Q: {question2}")
for r in result2:
    print(f"  {r['person1']} 创立了 {r['company1']}")
    print(f"  {r['person2']} 创立了 {r['company2']}")
    print(f"  两家公司关系: {r['relation']}")

In [ ]:
# ✅ 路径发现：两个实体之间的连接路径
result3 = graph.query("""
    MATCH path = shortestPath(
        (a:Person {name: '马云'})-[*]-(b:Product {name: '微信'})
    )
    RETURN [n IN nodes(path) | 
        CASE WHEN n:Person THEN 'Person:' + n.name
             WHEN n:Company THEN 'Company:' + n.name
             WHEN n:Product THEN 'Product:' + n.name
             WHEN n:City THEN 'City:' + n.name
             ELSE n.name END
    ] AS path_nodes
""")

print("马云 → 微信 的最短路径：")
for r in result3:
    print(" → ".join(r['path_nodes']))

### 刚才发生了什么？

| 问题类型 | 纯向量 RAG | 图查询 |
|---------|-----------|-------|
| 同城公司 | 需要检索多个文档 + LLM 推理 | 一次 `MATCH` 精确返回 |
| 多跳关系 | 可能漏掉关键信息 | 模式匹配直达目标 |
| 路径发现 | 几乎无法实现 | `shortestPath` 一行搞定 |

但图查询也有局限——它需要**精确知道要查什么**。如果用户的问题是模糊的自然语言（比如"阿里巴巴最近怎么样"），纯 Cypher 就不知道该匹配什么模式了。

这就引出了下一节的核心思路：**先让 LLM 从问题中提取实体，再用实体去查图**。

---

## 3. 实体提取驱动的图查询

核心流程：

```
用户问题 → LLM 提取实体 → 用实体查询图的邻域 → 图信息作为上下文 → LLM 生成回答
```

这是 GraphRAG 最基础也最实用的模式。

In [ ]:
# 第一步：从问题中提取实体
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

entity_prompt = ChatPromptTemplate.from_template(
    "从以下问题中提取关键实体名称（公司、人物、产品、城市），"
    "用逗号分隔，只输出实体名称，不要输出其他内容。\n\n"
    "问题：{question}"
)

entity_chain = entity_prompt | llm | StrOutputParser()

# 测试实体提取
question = "腾讯和阿里巴巴各自的主要产品是什么？"
entities = entity_chain.invoke({"question": question})
print(f"问题: {question}")
print(f"提取的实体: {entities}")

In [ ]:
# 第二步：查询实体的图邻域
entity_list = [e.strip() for e in entities.split(",")]

graph_context_parts = []
for entity in entity_list:
    result = graph.query(
        """
        MATCH (n)-[r]-(m)
        WHERE n.name = $entity
        RETURN n.name AS source,
               type(r) AS relation,
               m.name AS target,
               labels(m)[0] AS target_type
        LIMIT 10
        """,
        params={"entity": entity},
    )
    for r in result:
        graph_context_parts.append(
            f"{r['source']} --[{r['relation']}]--> {r['target']}（{r['target_type']}）"
        )

graph_context = "\n".join(graph_context_parts)
print("图上下文：")
print(graph_context)

### 刚才发生了什么？

我们用了一个非常简单但有效的策略：

1. **实体提取**：让 LLM 从自然语言问题中识别出"腾讯"和"阿里巴巴"
2. **图邻域查询**：对每个实体，查找它在图中的所有直接关系（一跳邻居）
3. **组装上下文**：把图关系转换为 `source --[relation]--> target` 格式的文本

这样 LLM 就能拿到**结构化的关系信息**来回答问题了。

In [ ]:
# 第三步：结合图上下文生成回答
qa_prompt = ChatPromptTemplate.from_template(
    "基于以下知识图谱信息回答问题。\n\n"
    "图谱信息：\n{graph_context}\n\n"
    "问题：{question}\n\n回答："
)

answer = (qa_prompt | llm | StrOutputParser()).invoke({
    "graph_context": graph_context,
    "question": question,
})
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
# 再测试几个问题
test_questions = [
    "张一鸣创立了什么公司？",
    "深圳有哪些科技公司？",
    "抖音和TikTok是哪个公司的产品？",
]

for q in test_questions:
    # 提取实体
    ents = entity_chain.invoke({"question": q})
    ent_list = [e.strip() for e in ents.split(",")]
    
    # 查图
    parts = []
    for entity in ent_list:
        result = graph.query(
            "MATCH (n)-[r]-(m) WHERE n.name = $entity "
            "RETURN n.name AS source, type(r) AS relation, m.name AS target LIMIT 10",
            params={"entity": entity},
        )
        parts.extend([f"{r['source']} --[{r['relation']}]--> {r['target']}" for r in result])
    
    ctx = "\n".join(parts) if parts else "未找到相关图谱信息"
    
    # 生成回答
    ans = (qa_prompt | llm | StrOutputParser()).invoke({
        "graph_context": ctx,
        "question": q,
    })
    print(f"Q: {q}")
    print(f"A: {ans}\n")

### 常见问题

**Q：实体提取不准怎么办？比如用户输入了"马爸爸"，但图里是"马云"。**

A：几种解决方案：
1. 在提取 prompt 中加入已知实体列表作为参考
2. 提取后用模糊匹配（如 Levenshtein 距离）在图中搜索
3. 在 Neo4j 中建全文索引，支持模糊搜索

**Q：只查一跳邻居够吗？**

A：对大多数问题够用。如果需要多跳推理，可以在 Cypher 中用 `(n)-[*1..3]-(m)` 控制遍历深度，但要注意结果膨胀问题。

---

## 4. 混合检索：向量 + 图结构

实体提取 + 图查询解决了关系推理问题，但对于**描述性问题**（"阿里巴巴的企业文化是什么"），文本文档可能包含更丰富的信息。

最佳策略：**两个都用**。把向量检索的语义匹配能力和图查询的结构化推理能力结合起来。

In [ ]:
def hybrid_rag(question):
    """混合 RAG：向量检索 + 图检索"""
    
    # === 路径 A：向量检索 ===
    vector_docs = retriever.invoke(question)
    vector_context = "\n".join([d.page_content for d in vector_docs])
    
    # === 路径 B：实体提取 + 图检索 ===
    entities = entity_chain.invoke({"question": question})
    entity_list = [e.strip() for e in entities.split(",")]
    
    graph_parts = []
    for entity in entity_list:
        result = graph.query(
            "MATCH (n)-[r]-(m) WHERE n.name = $entity OR n.id = $entity "
            "RETURN n.name AS s, type(r) AS r, m.name AS t LIMIT 5",
            params={"entity": entity},
        )
        graph_parts.extend([f"{r['s']} -[{r['r']}]-> {r['t']}" for r in result])
    
    graph_context = "\n".join(graph_parts) if graph_parts else "无图谱信息"
    
    # === 合并上下文，交给 LLM ===
    combined_prompt = ChatPromptTemplate.from_template(
        "基于以下信息回答问题。综合两个来源的信息给出完整回答。\n\n"
        "【文档内容】\n{vector_context}\n\n"
        "【知识图谱】\n{graph_context}\n\n"
        "问题：{question}\n回答："
    )
    
    return (combined_prompt | llm | StrOutputParser()).invoke({
        "vector_context": vector_context,
        "graph_context": graph_context,
        "question": question,
    })

print("hybrid_rag 函数定义完成")

In [ ]:
# 测试混合 RAG
test_questions = [
    "腾讯是谁创立的？在哪个城市？",
    "和华为在同一城市的公司有哪些？",
    "阿里巴巴有哪些主要产品？",
    "字节跳动和腾讯之间是什么关系？",
]

for q in test_questions:
    answer = hybrid_rag(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

### 刚才发生了什么？

混合 RAG 同时走两条路：

```
                    ┌─→ 向量检索 → 文档片段 ─┐
用户问题 ──→ │                              │──→ LLM 生成回答
                    └─→ 实体提取 → 图查询 ──┘
```

两个来源的信息互补：
- 文档片段提供了**自然语言描述**和**细节信息**
- 图查询提供了**精确的结构化关系**

LLM 综合两者给出更完整、更准确的回答。

---

## 5. 三种检索策略对比

我们用同一个问题测试三种方案，直观感受差异。

In [ ]:
def vector_only_rag(question):
    """纯向量 RAG"""
    context_docs = retriever.invoke(question)
    context_text = "\n".join([d.page_content for d in context_docs])
    prompt = ChatPromptTemplate.from_template(
        "基于以下内容回答问题。如果内容不足以准确回答，请说明。\n\n"
        "内容：{context}\n\n问题：{question}"
    )
    return (prompt | llm | StrOutputParser()).invoke({
        "context": context_text, "question": question
    })


def graph_only_rag(question):
    """纯图查询 RAG"""
    entities = entity_chain.invoke({"question": question})
    entity_list = [e.strip() for e in entities.split(",")]
    
    parts = []
    for entity in entity_list:
        result = graph.query(
            "MATCH (n)-[r]-(m) WHERE n.name = $entity "
            "RETURN n.name AS s, type(r) AS r, m.name AS t LIMIT 10",
            params={"entity": entity},
        )
        parts.extend([f"{r['s']} -[{r['r']}]-> {r['t']}" for r in result])
    
    ctx = "\n".join(parts) if parts else "未找到相关图谱信息"
    prompt = ChatPromptTemplate.from_template(
        "基于以下知识图谱信息回答问题。\n\n"
        "图谱信息：\n{context}\n\n问题：{question}\n回答："
    )
    return (prompt | llm | StrOutputParser()).invoke({
        "context": ctx, "question": question
    })


print("三种 RAG 函数定义完成")

In [ ]:
# 对比测试
comparison_question = "和腾讯在同一个城市的科技公司有哪些？它们各自有什么产品？"

print(f"问题: {comparison_question}")
print("=" * 60)

print("\n【纯向量 RAG】")
print(vector_only_rag(comparison_question))

print("\n【纯图查询 RAG】")
print(graph_only_rag(comparison_question))

print("\n【混合 GraphRAG】")
print(hybrid_rag(comparison_question))

### 对比总结

| 策略 | 实现方式 | 擅长 | 不擅长 |
|------|---------|------|--------|
| 纯向量 RAG | FAISS + 语义相似度 | 模糊语义匹配、描述性问题 | 精确关系推理、多跳查询 |
| 纯图查询 RAG | 实体提取 + Cypher 遍历 | 结构化关系查询、路径发现 | 自由文本理解、模糊问题 |
| GraphRAG（混合） | 向量 + 图遍历 | 两者兼顾，互补增强 | 实现复杂度较高 |

**选择建议：**

- 如果你的数据是**纯文档**（PDF、网页），纯向量 RAG 就够了
- 如果你的数据有**明确的实体关系**（知识图谱、组织架构），优先用图查询
- 如果**两者都有**，或者问题类型不确定，用混合 GraphRAG 最稳妥

---

## 6. 引用来源追踪

在生产环境中，用户不仅想要回答，还想知道**答案从哪来的**。混合 RAG 需要同时追踪向量检索和图查询两个来源。

In [ ]:
def hybrid_rag_with_sources(question):
    """带引用来源追踪的混合 RAG"""
    sources = {"vector": [], "graph": []}
    
    # === 向量检索 ===
    vector_docs = retriever.invoke(question)
    vector_context = "\n".join([d.page_content for d in vector_docs])
    for doc in vector_docs:
        sources["vector"].append({
            "content": doc.page_content[:80] + "...",
            "metadata": doc.metadata,
        })
    
    # === 图检索 ===
    entities = entity_chain.invoke({"question": question})
    entity_list = [e.strip() for e in entities.split(",")]
    
    graph_parts = []
    for entity in entity_list:
        result = graph.query(
            "MATCH (n)-[r]-(m) WHERE n.name = $entity "
            "RETURN n.name AS s, type(r) AS rel, m.name AS t, "
            "labels(n)[0] AS s_type, labels(m)[0] AS t_type LIMIT 5",
            params={"entity": entity},
        )
        for r in result:
            triple = f"{r['s']} -[{r['rel']}]-> {r['t']}"
            graph_parts.append(triple)
            sources["graph"].append({
                "triple": triple,
                "source_type": r["s_type"],
                "target_type": r["t_type"],
            })
    
    graph_context = "\n".join(graph_parts) if graph_parts else "无图谱信息"
    
    # === 生成带引用的回答 ===
    combined_prompt = ChatPromptTemplate.from_template(
        "基于以下信息回答问题。请在回答中标注信息来源（[文档] 或 [图谱]）。\n\n"
        "【文档内容】\n{vector_context}\n\n"
        "【知识图谱】\n{graph_context}\n\n"
        "问题：{question}\n"
        "回答（请标注信息来源）："
    )
    
    answer = (combined_prompt | llm | StrOutputParser()).invoke({
        "vector_context": vector_context,
        "graph_context": graph_context,
        "question": question,
    })
    
    return answer, sources

In [ ]:
# 测试带来源的混合 RAG
question = "华为有哪些产品？在哪个城市？"
answer, sources = hybrid_rag_with_sources(question)

print(f"Q: {question}")
print(f"A: {answer}")
print("\n--- 来源追踪 ---")
print("\n向量检索来源：")
for s in sources["vector"]:
    print(f"  - {s['content']}")
    print(f"    metadata: {s['metadata']}")
print("\n图谱查询来源：")
for s in sources["graph"]:
    print(f"  - {s['triple']}")

### 刚才发生了什么？

我们在混合 RAG 函数中增加了来源追踪：

1. **向量来源**：记录检索到的文档片段和元数据
2. **图谱来源**：记录查询到的三元组（`source -[relation]-> target`）
3. **回答标注**：在 prompt 中要求 LLM 标注每条信息的来源

这在生产环境中非常重要——用户可以验证回答的可靠性，开发者可以调试检索效果。

### 常见问题

**Q：图检索和向量检索的结果冲突了怎么办？**

A：一般来说图谱数据更可靠（因为是结构化的），可以在 prompt 中加一句"当两个来源冲突时，优先参考知识图谱"。更好的做法是在数据入库时就保证两边一致。

**Q：实体提取调用了一次 LLM，最终生成又调用一次，成本会不会太高？**

A：实体提取的输入输出都很短，token 消耗很少。相比回答质量的提升，这个成本完全值得。如果追求极致性能，可以用小模型做实体提取、大模型做最终生成。

---

## 总结

本章学习了：

- 纯向量 RAG 在关系推理上的局限
- 基于图结构的检索（精确关系查询）
- 实体提取驱动的图查询流程（问题 → 实体 → 图邻域 → 上下文）
- 混合检索策略（向量 + 图遍历）
- 三种检索策略的适用场景对比
- 引用来源追踪的实现

核心要点：**向量检索和图检索不是非此即彼的关系，而是互补的**。向量擅长模糊语义匹配，图擅长精确关系推理。GraphRAG 把两者的优势结合起来，让 LLM 能同时利用非结构化文本和结构化知识。

---

## 下一章预告

**第七章：图驱动 Agent 与生产实践** —— 最后一章将图查询和 GraphRAG 封装为 Agent 工具，让 Agent 自主决定何时查图、何时查向量。同时介绍图维护、性能优化等生产实践。